# Fine-tuning Answer-Aware CrossEncoder Reranker on TechQA

**BEFORE YOU RUN**: Settings → Accelerator → GPU T4

**Improvement over the previous notebook:**

Instead of (question, doc) pairs, we use (question, answer_snippet, doc) — three separate inputs. This way, the reranker learns that a document **containing** the answer is correct, even if it doesn’t immediately resemble a troubleshooting guide.

In [1]:
!pip install -q sentence-transformers torch

In [2]:
import json
import random
import os
import torch
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from torch.utils.data import DataLoader

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found!')

random.seed(42)

GPU available: True
GPU: Tesla T4


In [3]:
BASE = '/kaggle/input/datasets/yiannisman/techqa-dataset/TechQA'

TECHNOTES_PATH = f'{BASE}/technote_corpus/full_technote_collection.sections.json'
TRAIN_QA_PATH  = f'{BASE}/training_and_dev/training_Q_A.json'
DEV_QA_PATH    = f'{BASE}/training_and_dev/dev_Q_A.json'

for path in [TECHNOTES_PATH, TRAIN_QA_PATH, DEV_QA_PATH]:
    exists = os.path.exists(path)
    status = 'CHECK' if exists else 'FAIL'
    size   = f'{os.path.getsize(path)/1024/1024:.1f} MB' if exists else 'NOT FOUND'
    print(f'{status} {os.path.basename(path)}: {size}')

CHECK full_technote_collection.sections.json: 6124.5 MB
CHECK training_Q_A.json: 1.1 MB
CHECK dev_Q_A.json: 0.5 MB


In [4]:
print('Loading Technotes...')
technotes = {}

with open(TECHNOTES_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

for doc_id, doc_data in raw.items():
    technotes[doc_id] = doc_data.get('text', '')

print(f'{len(technotes)} Loaded Technotes.')

Loading Technotes...
801998 Loaded Technotes.


In [5]:
def load_qa(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('Loading Q&A pairs...')
train_data = load_qa(TRAIN_QA_PATH)
dev_data   = load_qa(DEV_QA_PATH)

all_qa = [
    item for item in train_data + dev_data
    if item.get('ANSWERABLE') == 'Y'
    and item.get('DOCUMENT')
    and item.get('DOCUMENT') in technotes
    and item.get('ANSWER')   # <- ANSWER field needed
    and str(item.get('ANSWER')).strip()  # <- Only if blank
]

print(f'Train questions:                {len(train_data)}')
print(f'Dev questions:                  {len(dev_data)}')
print(f'Answerable με valid doc+answer: {len(all_qa)}')

Loading Q&A pairs...
Train questions:                600
Dev questions:                  310
Answerable με valid doc+answer: 608


In [6]:
# -- Creation Training Pairs ---------------------------------------------------

NUM_NEGATIVES    = 4
all_technote_ids = list(technotes.keys())

train_samples = []
skipped       = 0

for item in all_qa:
    question      = str(item.get('QUESTION_TEXT') or item.get('QUESTION_TITLE') or '').strip()
    correct_id    = item['DOCUMENT']
    correct_doc   = technotes[correct_id]

    if not question:
        skipped += 1
        continue

    # IMPORTANT: The query is ONLY the question. No answer, no HyDE.
    # We teach the model the relationship: "What the user is asking" -> "Which document solves the problem"
    pure_query = question 

    # Positive pair
    train_samples.append(InputExample(
        texts=[pure_query, correct_doc],
        label=1.0
    ))

    # Negative pairs
    neg_ids = random.sample(
        [tid for tid in all_technote_ids if tid != correct_id],
        NUM_NEGATIVES
    )
    for neg_id in neg_ids:
        train_samples.append(InputExample(
            texts=[pure_query, technotes[neg_id]],
            label=0.0
        ))

random.shuffle(train_samples)

valid_q = len(all_qa) - skipped
print(f'Total training samples: {len(train_samples)}')
print(f'  Positive pairs: {valid_q}')
print(f'  Negative pairs: {valid_q * NUM_NEGATIVES}')

Total training samples: 3040
  Positive pairs: 608
  Negative pairs: 2432


In [7]:
split_idx   = int(len(train_samples) * 0.9)
train_split = train_samples[:split_idx]
val_split   = train_samples[split_idx:]

print(f'Training samples:   {len(train_split)}')
print(f'Validation samples: {len(val_split)}')

Training samples:   2736
Validation samples: 304


In [8]:
MODEL_NAME  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
OUTPUT_PATH = '/kaggle/working/techqa-reranker-v2'
EPOCHS      = 3
BATCH_SIZE  = 16

print(f'Loading base model: {MODEL_NAME}...')
model = CrossEncoder(MODEL_NAME, num_labels=1, max_length=512)

train_dataloader = DataLoader(train_split, shuffle=True, batch_size=BATCH_SIZE)
warmup_steps     = int(len(train_dataloader) * EPOCHS * 0.1)

print(f'Training steps/epoch: {len(train_dataloader)}')
print(f'Warmup steps:         {warmup_steps}')
print(f'Start fine-tuning για {EPOCHS} epochs...')

model.fit(
    train_dataloader=train_dataloader,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    save_best_model=False
)

print(f'\nFine-tuning Completed!')

Loading base model: cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Training steps/epoch: 171
Warmup steps:         51
Start fine-tuning για 3 epochs...


Token indices sequence length is longer than the specified maximum sequence length for this model (563 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss



Fine-tuning Completed!


In [9]:
print(f'Saving Model -> {OUTPUT_PATH}...')
os.makedirs(OUTPUT_PATH, exist_ok=True)
model.save(OUTPUT_PATH)

print(f'\nFiles:')
for fname in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(os.path.join(OUTPUT_PATH, fname)) / (1024*1024)
    print(f'  {fname}: {size:.1f} MB')

Saving Model -> /kaggle/working/techqa-reranker-v2...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Files:
  README.md: 0.1 MB
  model.safetensors: 86.7 MB
  tokenizer_config.json: 0.0 MB
  config.json: 0.0 MB
  tokenizer.json: 0.7 MB
